# 00 – Study Area Exploration via PFAF_ID

**Purpose:** Defining a study area by selecting a Pfafstetter Level-6 basin from the global SWORD dataset.
This notebook is the single entry point for all subsequent analysis notebooks.
**Study area and PFAF_IDs are defined centrally in _00_config.py**. So manually steps are required, the notebook is fully automated.

**OUTPUT:** 
Overview of certain river system dimension.

**Concept:**
SWORD encodes the Pfafstetter Level-6 basin code directly in the first 6 digits of each `reach_id`:
```
reach_id format: CBBBBBRRRRT
  C      = Continent digit (first digit of Pfafstetter code)
  BBBBB  = Remaining Pfafstetter digits up to Level 6
  --> CBBBBB (first 6 digits) = Pfafstetter Level-6 basin code = PFAF_ID
  RRR    = Reach number within basin
  T      = Reach type (1=river, 3=lake, 4=dam, 5=unreliable, 6=ghost)
```
This means no spatial join is needed to assign a basin to a reach.
The PFAF_ID is extracted directly from `reach_id` as a new column.

**Workflow:**
```
0.0 Imports
0.1 Configuration  <-- only cell that needs editing between runs
0.2 Load global SWORD
0.3 Extract PFAF_ID column
0.4 Filter to study area
0.5 Validate & inspect
0.6 Save clipped SWORD (only for testing single rivers)
0.7 Interactive map
```

**Output:** `sword_{STUDY_AREA}_clip.gpkg` – clipped SWORD reaches for the chosen study area,
with `PFAF_ID` column added. This file is the input for Notebook 02 (Join).

---
## 0.0 | Imports

In [ ]:
import geopandas as gpd
import pandas as pd
import os
import sys

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT

In [ ]:
# Study area and PFAF_IDs are defined centrally in _00_config.py
# Change the river there -- no edits needed in this notebook.
from _00_config import STUDY_AREA, PFAF_IDS, PFAF_LEVEL_DIGITS

---
## 0.1 | Configuration

To find the right PFAF_IDs for a river:
- Look up the first 6 digits of any known `reach_id` for that river (e.g. from Notebook 01)
- Or browse HydroSHEDS / HydroBASINS Level 6 to identify the relevant basin codes
- A river may span multiple Level-6 basins (e.g. a large river with major tributaries)
  --> list all relevant codes in `PFAF_IDS`


**Examples:**

**Study area and PFAF_IDs are defined centrally in _00_config.py**

```python
# Naryn River (Kyrgyzstan / Uzbekistan)
STUDY_AREA = "naryn"
PFAF_IDS   = [462193]

# Elbe River (Germany / Czech Republic) -- placeholder, verify before use
STUDY_AREA = "elbe"
PFAF_IDS   = [231647, 231648]  # example: may span multiple Level-6 basins

# Amazon main stem + major tributaries -- large rivers need more codes
STUDY_AREA = "amazon"
PFAF_IDS   = [62244, 62292, 62262]  # placeholder
```

In [ ]:
# ============================================================
# PATHS -- all derived automatically, do not edit below
# ============================================================

# Mapping: first digit of Pfafstetter code --> SWORD continent file prefix
# The first digit of PFAF_ID encodes the continent in the Pfafstetter system
PFAF_CONTINENT_MAP = {
    1: "af",   # Africa
    2: "eu",   # Europe
    3: "as",   # Siberia / North Asia (Lena, Ob, Yenisei)
    4: "as",   # Asia (Naryn, Mekong, Indus, ...)
    5: "oc",   # Australia / Oceania
    6: "sa",   # South America
    7: "na",   # North America
    8: "na",   # North America (Arctic / Canada)
    9: "na",   # Arctic
}

# All available SWORD continent files
SWORD_FILES = {
    "as": os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "as_sword_reaches_v17b.gpkg"),
    "af": os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "af_sword_reaches_v17b.gpkg"),
    "eu": os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "eu_sword_reaches_v17b.gpkg"),
    "na": os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "na_sword_reaches_v17b.gpkg"),
    "sa": os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "sa_sword_reaches_v17b.gpkg"),
    "oc": os.path.join(DATA_RAW, "0_data", "SWOT_river_database_SWORD", "oc_sword_reaches_v17b.gpkg"),
}

# Reach types: 1=river, 3=lake on river, 4=dam/waterfall, 5=unreliable topology, 6=ghost reach
# Type 3 (lake on river) is kept because wide/braided river sections are classified as
# lake-type in SWORD due to their surface appearance in SWOT satellite measurements.
# Excluding type 3 would systematically remove the most morphologically interesting reaches
# (channel widening, alluvial fans, braided sections).
REACH_TYPES_TO_KEEP = [1, 3]

# Automatically determine which continent file to load based on PFAF_IDS
# All PFAF_IDs in one study area must be on the same continent -- warn if not
_continent_codes = set(PFAF_CONTINENT_MAP.get(int(str(p)[0])) for p in PFAF_IDS)

if len(_continent_codes) > 1:
    # Cross-continent rivers (very rare, e.g. Nile spanning Africa/Asia)
    # would need merging of multiple files -- not yet supported
    raise ValueError(
        f"PFAF_IDs span multiple continents {_continent_codes}. "
        f"Multi-continent rivers are not yet supported -- split into separate study areas."
    )

CONTINENT = _continent_codes.pop()
IN_SWORD = SWORD_FILES[CONTINENT]

# Output paths
OUT_SWORD_CLIP = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_clip.gpkg")
OUT_MAP        = os.path.join(DATA_OUTPUT,    f"00_{STUDY_AREA}_study_area_map.html")

# ============================================================
# SUMMARY
# ============================================================
print(f"Study area  : {STUDY_AREA}")
print(f"PFAF_IDs    : {PFAF_IDS}")
print(f"Continent   : {CONTINENT}  (auto-detected from PFAF_ID)")
print(f"Reach types : {REACH_TYPES_TO_KEEP}")
print(f"Input       : {IN_SWORD}")
print(f"Output      : {OUT_SWORD_CLIP}")
print()
print(f"Input exists: {os.path.exists(IN_SWORD)}")

---
## 0.2 | Load global SWORD

The global SWORD dataset contains all reaches worldwide.

**Note on performance:** Loading the full global SWORD can be slow (minutes depending on hardware).
If working with multiple study areas frequently, pre-splitting SWORD by continent
(first digit of reach_id = continent code) should ne done to reduce load times.

In [ ]:
print("Loading AOI SWORD dataset...")
aoi = gpd.read_file(IN_SWORD)

print(f"AOI SWORD loaded: {len(aoi):,} reaches")
print(f"CRS: {aoi.crs}")
print(f"Columns: {aoi.columns.tolist()}")

---
## 0.3 | Extract PFAF_ID column

The Pfafstetter Level-6 code is encoded in the first 6 digits of `reach_id`.
It will be extracted as a dedicated integer column `PFAF_ID` so it can be used
for filtering, grouping, and downstream joins.

In [ ]:
# Extract PFAF code at the chosen level from reach_id
aoi["PFAF_ID"] = (
    aoi["reach_id"]
    .astype(str)
    .str[:PFAF_LEVEL_DIGITS]   # configurable: 5 = Level 5, 6 = Level 6
    .astype(int)
)

# Also extract reach type from the last digit of reach_id for filtering.
# Type: 1=river, 3=lake on river, 4=dam/waterfall, 5=unreliable topology, 6=ghost reach
aoi["reach_type"] = (
    aoi["reach_id"]
    .astype(str)
    .str[-1]       # last character
    .astype(int)
)

print(f"PFAF_ID column added. Unique basins in global dataset: {aoi['PFAF_ID'].nunique():,}")
print(f"\nReach type distribution (global):")
print(aoi["reach_type"].value_counts().sort_index())

---
## 0.4 | Filter to study area

Filtering the global dataset to the PFAF_IDs defined in Section 0.1.

In [ ]:
# Step 1: Filter by PFAF_ID (study area definition)
mask_basin = aoi["PFAF_ID"].isin(PFAF_IDS)

# Step 2: Filter by reach type (keep only valid river reaches by default)
mask_type = aoi["reach_type"].isin(REACH_TYPES_TO_KEEP)

# Apply both filters
sword_clip = aoi[mask_basin & mask_type].copy()

# Reset index for clean downstream processing
sword_clip = sword_clip.reset_index(drop=True)

# -------- Summary --------
print(f"Reaches in study area  : {len(sword_clip):,}")
print(f"Reaches excluded (type): {mask_basin.sum() - len(sword_clip):,}")
print(f"PFAF_IDs found         : {sorted(sword_clip['PFAF_ID'].unique())}")
print(f"Reach types kept       : {sorted(sword_clip['reach_type'].unique())}")

# Sanity check: warn if any requested PFAF_ID returned zero reaches
for pfaf in PFAF_IDS:
    n = (sword_clip["PFAF_ID"] == pfaf).sum()
    if n == 0:
        print(f"  WARNING: PFAF_ID {pfaf} matched 0 reaches -- check if code is correct")
    else:
        print(f"  PFAF_ID {pfaf}: {n} reaches")

---
## 0.5 | Validate & inspect

Quick sanity checks before saving. These should catch obvious problems early
(wrong PFAF_ID, missing geometry, unexpected reach counts).

In [ ]:
# --- Geometry checks --------
n_null_geom = sword_clip.geometry.isnull().sum()
n_empty_geom = sword_clip.geometry.is_empty.sum()

print("=== Geometry ===")
print(f"  Null geometries : {n_null_geom}")
print(f"  Empty geometries: {n_empty_geom}")

# --- Key attribute checks --------
print("\n=== Key attributes ===")
for col in ["reach_id", "dist_out", "reach_len", "slope", "n_chan_mod", "facc"]:
    if col in sword_clip.columns:
        n_null = sword_clip[col].isnull().sum()
        pct = 100 * n_null / len(sword_clip)
        print(f"  {col:<20}: {n_null:>4} null ({pct:.1f}%)")
    else:
        print(f"  {col:<20}: COLUMN NOT FOUND")

# -------- Spatial extent --------
print("\n=== Spatial extent ===")
bounds = sword_clip.total_bounds  # (minx, miny, maxx, maxy)
print(f"  Lon: {bounds[0]:.4f} to {bounds[2]:.4f}")
print(f"  Lat: {bounds[1]:.4f} to {bounds[3]:.4f}")

# -------- Reach length distribution --------
print("\n=== Reach length (m) ===")
if "reach_len" in sword_clip.columns:
    print(sword_clip["reach_len"].describe().round(1).to_string())

---
## 0.6 | Save clipped SWORD

Output: `sword_{STUDY_AREA}_clip.gpkg` in the processed data folder.
This file is the single input for Notebook 02 (Join).

In [ ]:
# Create output directory if it does not exist yet
os.makedirs(DATA_PROCESSED, exist_ok=True)

# Save as GeoPackage (preserves CRS and geometry type reliably)
sword_clip.to_file(OUT_SWORD_CLIP, driver="GPKG")

print(f"Saved: {OUT_SWORD_CLIP}")
print(f"Reaches saved: {len(sword_clip):,}")
print(f"File size: {os.path.getsize(OUT_SWORD_CLIP) / 1024:.1f} KB")

---
## 0.7 | Interactive map

Visual verification that the right reaches were selected.
Colored by `PFAF_ID` so it is visible if multiple basins
are correctly distinguished, or if an unexpected basin crept in.

In [ ]:
import webbrowser

# Interactive map colored by PFAF_ID
# If only one PFAF_ID is selected, colors by reach_id instead for more detail
color_col = "PFAF_ID" if sword_clip["PFAF_ID"].nunique() > 1 else "reach_id"

m = sword_clip.explore(
    column=color_col,
    cmap="tab10",
    tooltip=["reach_id", "PFAF_ID", "reach_type", "dist_out", "reach_len", "slope"],
    tiles="CartoDB positron",
    legend=True,
    style_kwds={"weight": 2, "opacity": 0.8}
)

os.makedirs(DATA_OUTPUT, exist_ok=True)
m.save(OUT_MAP)
webbrowser.open(OUT_MAP)
print(f"Map saved: {OUT_MAP}")